# CELL 1: Install (if needed)

In [62]:
# Run only if sklearn/joblib not installed
# !pip install scikit-learn joblib

# CELL 2: Imports

In [63]:
import numpy as np
import joblib

from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


# CELL 3: Load Feature Extractor (GoogLeNet-style)

In [64]:
base_model = InceptionV3(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

x = base_model.output
x = GlobalAveragePooling2D()(x)

feature_extractor = Model(inputs=base_model.input, outputs=x)

print(" Feature extractor ready :)")

 Feature extractor ready :)


# CELL 4: Data Generators (NO augmentation)

In [65]:
datagen = ImageDataGenerator(rescale=1./255)

train_generator = datagen.flow_from_directory(
    'dataset/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

validation_generator = datagen.flow_from_directory(
    'dataset/validation',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

test_generator = datagen.flow_from_directory(
    'dataset/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

print("Classes:", train_generator.class_indices)

Found 1145 images belonging to 6 classes.
Found 267 images belonging to 6 classes.
Found 220 images belonging to 6 classes.
Classes: {'A1-Walking': 0, 'A2-Sitting-down': 1, 'A3-StandUp': 2, 'A4-PickObject': 3, 'A5-DrinkWater': 4, 'A6-Fall': 5}


# CELL 5: Feature Extraction Function

In [66]:
def extract_features(generator, model):
    features = []
    labels = []

    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        f_batch = model.predict(x_batch, verbose=0)

        features.append(f_batch)
        labels.append(np.argmax(y_batch, axis=1))

    features = np.vstack(features)
    labels = np.hstack(labels)

    return features, labels

# CELL 6: Extract Features

In [67]:
print("Extracting TRAIN features...")
X_train, y_train = extract_features(train_generator, feature_extractor)

print("Extracting VALIDATION features...")
X_val, y_val = extract_features(validation_generator, feature_extractor)

print("Extracting TEST features...")
X_test, y_test = extract_features(test_generator, feature_extractor)

print("Shapes:")
print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Extracting TRAIN features...
Extracting VALIDATION features...
Extracting TEST features...
Shapes:
Train: (1145, 2048)
Val: (267, 2048)
Test: (220, 2048)


# CELL 7: Normalize Features

In [68]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(" Features normalized _/_/")

 Features normalized _/_/


# CELL 8: Train SVM

In [69]:
print("Training SVM...")

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.001],
    'kernel': ['rbf']
}

grid = GridSearchCV(
    SVC(probability=True),
    param_grid,
    cv=3,
    verbose=2,
    n_jobs=-1   # uses all CPU cores (faster)
)

grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)

svm_model = grid.best_estimator_

print("_/ _/Best SVM model ready")

Training SVM...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Params: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}
_/ _/Best SVM model ready


# CELL 9: Validation Evaluation (for tuning)

In [70]:
y_val_pred = svm_model.predict(X_val)

val_acc = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", val_acc)

Validation Accuracy: 0.8239700374531835


# CELL 10: Test Evaluation

In [71]:
y_test_pred = svm_model.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)

print("_/_/ Final Test Accuracy:", test_acc)

print("\nClassification Report:\n")
print(classification_report(y_test, y_test_pred))

_/_/ Final Test Accuracy: 0.7181818181818181

Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.95      0.97        40
           1       0.76      0.62      0.68        40
           2       0.58      0.55      0.56        40
           3       0.63      0.78      0.70        40
           4       0.76      0.55      0.64        40
           5       0.61      1.00      0.75        20

    accuracy                           0.72       220
   macro avg       0.72      0.74      0.72       220
weighted avg       0.73      0.72      0.72       220



# CELL 11: Save Models

In [72]:
joblib.dump(svm_model, "googlenet_svm_model.pkl")
joblib.dump(scaler, "googlenet_svm_scaler.pkl")

print("_/_/ Models saved")

_/_/ Models saved


# CELL 12: Single Image Prediction

In [73]:
from tensorflow.keras.preprocessing import image

# Load models
svm_model = joblib.load("googlenet_svm_model.pkl")
scaler = joblib.load("googlenet_svm_scaler.pkl")

# Reverse class mapping
class_indices = train_generator.class_indices
index_to_class = {v: k for k, v in class_indices.items()}

# Load image
img_path = "dataset/test/A2-Sitting-down/235.png"
img = image.load_img(img_path, target_size=(224,224))

img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# Extract features
features = feature_extractor.predict(img_array)

# Scale
features = scaler.transform(features)

# Predict
pred = svm_model.predict(features)[0]
confidence = np.max(svm_model.predict_proba(features))

print("Predicted Class:", index_to_class[pred])
print("Confidence:", confidence)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step
Predicted Class: A2-Sitting-down
Confidence: 0.9958985893439868
